In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col

In [0]:
dbutils.widgets.text("catalog", "azdeusdbricks")

In [0]:
catalog = dbutils.widgets.get("catalog")
volume_path = "/Volumes/azdeusdbricks/landing/source_files"


In [0]:
df = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaLocation", "/Volumes/azdeusdbricks/bronze/raw_files/_checkpoint/schema")
    .load(volume_path)
)

In [0]:
df_enriched = (
    df.withColumn("_ingestTimeStamp", F.current_timestamp())
    .withColumn("sourceFileLocation", col("_metadata.file_path"))
)

In [0]:
query = (df_enriched.writeStream
    .option("checkpointLocation", "/Volumes/azdeusdbricks/bronze/raw_files/_checkpoint")
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(f"{catalog}.bronze.raw_files"))

query.awaitTermination()
spark.table(f"{catalog}.bronze.raw_files").display()